In [1]:
import $ivy.`org.typelevel::cats-core:2.3.0`
import $ivy.`org.typelevel::cats-effect:2.5.3`
import $plugin.$ivy.`org.typelevel::kind-projector:0.10.3`
import $ivy.`org.scalatest::scalatest:3.2.12`

import $ivy.$                               

import $ivy.$                                 

import $plugin.$                                          

import $ivy.$                                

In [2]:
import cats.implicits._, cats.syntax.all._, cats.data._, cats.Monad
import cats.effect.IO
import org.scalatest._
import org.scalatest.flatspec.AnyFlatSpec
import matchers.should._

import cats.implicits._, cats.syntax.all._, cats.data._, cats.Monad

import cats.effect.IO

import org.scalatest._

import org.scalatest.flatspec.AnyFlatSpec

import matchers.should._

# Auxiliary functions

For data-driven computations:

In [3]:
implicit class NextOp[A](a: A){
    
    def mapIf[B >: A](p: Boolean)(f: A => B): B = 
        if (p) f(a)
        else a
    
    def mapAs[B](f: A => B): B = 
        f(a)
}

defined class NextOp

For dictionaries:

In [4]:
implicit class UpdatedMap[K, V](m: Map[K, V]){
    
    def modifyOrAdd(k: K)(n: V, f: V => V): Map[K, V] = 
        if (!m.isDefinedAt(k)) m + (k -> n)
        else m.updated(k, f(m(k)))
    
    def modify(k: K)(f: V => V): Map[K, V] = 
        if (!m.isDefinedAt(k)) m
        else m.updated(k, f(m(k)))
    
    def containsSuch(k: K)(p: V => Boolean): Boolean = 
        m.get(k).map(p).getOrElse(false)
}

defined class UpdatedMap

IO-related:

In [5]:
def putStrLn(value: String) = IO(println(value))
def readLn(value: String) = IO(scala.io.StdIn.readLine(value))

defined function putStrLn
defined function readLn

In [6]:
// TODO: use iterateUntil _.isFinished
def iterate[A](times: Int)(from: IO[A])(cont: A => IO[A]): IO[A] = 
    if (times == 0) from
    else iterate(times-1)(from >>= cont)(cont)

defined function iterate

# Wordle Spec

In [7]:
type Word = String

defined type Word

In [8]:
sealed trait Tag
case object Green extends Tag
case object Yellow extends Tag
case object Gray extends Tag
case object Red extends Tag

defined trait Tag
defined object Green
defined object Yellow
defined object Gray
defined object Red

In [9]:
type TaggedWord = IndexedSeq[(Char, Tag)]

object TaggedWord{
    def guessed(w: TaggedWord): Boolean = 
        w.forall(_._2 == Green)
}

defined type TaggedWord
defined object TaggedWord

In [10]:
class TagSpec(tag: (Word, Word) => TaggedWord) extends AnyFlatSpec with Matchers{
    "matching characters" should "be detected" in {
        tag("word", "word") shouldBe Vector('w' -> Green, 'o' -> Green, 'r' -> Green, 'd' -> Green)
        tag("wor", "war") should matchPattern { case Vector(('w', Green), _, ('r', Green)) => }
    }
    
    "non-matching characters" should "be detected" in {
        tag("vos", "war") shouldBe Vector('v' -> Gray, 'o' -> Gray, 's' -> Gray)
        tag("wor", "war") should matchPattern { case Vector(_, ('o', Gray), _) => }
        tag("rrrr", "warr") should matchPattern { case Vector('r' -> Gray, ('r', Gray), _, _) => }
    }
    
    "misplaced characters" should "be detected" in {
        tag("rwa", "war") shouldBe Vector('r' -> Yellow, 'w' -> Yellow, 'a' -> Yellow)
    }
}

defined class TagSpec

# First version: _control_ driven

In [ ]:
case class Wordle(guess: String, attempts: List[TaggedWord]){
    def attempt(word: String): Wordle = 
        Wordle(guess, Wordle.TagState.tag(word, guess) :: attempts)
}

object Wordle{

    def initial(guess: String): Wordle = 
        Wordle(guess, List())
    
    def run(words: List[Word], guess: String): Wordle = 
        words.foldLeft(initial(guess))(_.attempt(_))

    case class TagState(maybeYellow: Map[Char, List[Int]], 
                        occurrences: Map[Char, Int], 
                        tagged: TaggedWord, 
                        index: Int)
    object TagState{
        
        def tag(word: String, guess: String): TaggedWord = 
            tagState(word, guess).runA(initial).value
        
        val initial = TagState(Map(), Map(), IndexedSeq(), 0)
    
        def tagState(word: String, guess: String): State[TagState, TaggedWord] = 
            word.zip(guess).toList.traverse{ case (c, g) => 
                State.modify[TagState]{ case TagState(maybeYellow, occurrences, tagged, index) => 
                    val matched = c == g
                    val matchedCurrent = occurrences.containsSuch(c)(_ > 0)
                    val matchedGuess = maybeYellow.containsSuch(g)(!_.isEmpty)

                    if (matched)
                        TagState(
                            maybeYellow, 
                            occurrences, 
                            tagged :+ (c -> Green),
                            index + 1)
                    else if (!matched && !matchedCurrent && !matchedGuess)
                        TagState(
                            maybeYellow.modifyOrAdd(c)(List(index), _ :+ index),
                            occurrences.modifyOrAdd(g)(1, _ + 1),
                            tagged :+ (c -> Gray),
                            index + 1)
                    else if (!matched && !matchedCurrent && matchedGuess)
                        TagState(
                            maybeYellow.modifyOrAdd(c)(List(index), _ :+ index).modify(g)(_.tail),
                            occurrences,
                            tagged.updated(maybeYellow(g).head, g -> Yellow) :+ (c -> Gray),
                            index + 1)
                    else if (!matched && matchedCurrent && !matchedGuess)
                        TagState(
                            maybeYellow,
                            occurrences.modify(c)(_ - 1).modifyOrAdd(g)(1, _ + 1),
                            tagged :+ (c -> Yellow),
                            index + 1)
                    else 
                        TagState(
                            maybeYellow.modify(g)(_.tail),
                            occurrences.modify(c)(_ - 1),
                            tagged.updated(maybeYellow(g).head, g -> Yellow) :+ (c -> Yellow),
                            index + 1)
                }//.inspect(println(_))
            
            }.inspect(_.tagged)
    }
        
}


In [ ]:
run(new TagSpec(Wordle.TagState.tag))

# Second version: _data_ driven 

In [ ]:
case class Wordle(guess: String, attempts: List[TaggedWord]){
    
    def attempt(word: String): Wordle = 
        Wordle(guess, Wordle.TagState.tag(word, guess) :: attempts)
    
    def isFinished: Boolean = 
        attempts.length >= 6 || attempts.headOption.map(TaggedWord.guessed).getOrElse(false)
}

object Wordle{

    def initial(guess: String): Wordle = 
        Wordle(guess, List())
    
    def run(words: List[Word], guess: String): Wordle = 
        words.foldLeft(initial(guess))(_.attempt(_))
    
    def console: IO[Wordle] = 
        readLn("Word to guess: ") >>= { guess: String => 
            iterate(6)(IO(Wordle.initial(guess))){ wordle: Wordle => 
                readLn("Your attempt: ") >>= { word: String => 
                    val next = wordle.attempt(word)
                    putStrLn(next.attempts.head.toString).as(next)
                }
            }
        }

    case class TagState(maybeYellow: Map[Char, List[Int]], 
                        occurrences: Map[Char, Int], 
                        tagged: TaggedWord, 
                        index: Int)
    object TagState{
        
        def tag(word: String, guess: String): TaggedWord = 
            tagState(word, guess).runA(initial).value
        
        val initial = TagState(Map(), Map(), IndexedSeq(), 0)
    
        def tagState(word: String, guess: String): State[TagState, TaggedWord] = 
            word.zip(guess).toList.traverse{
                case (c, g) => State.modify[TagState]{ 
                    case TagState(maybeYellow, occurrences, tagged, index)=> {
                        val matched = c == g
                        val matchedCurrent = occurrences.containsSuch(c)(_ > 0)
                        val matchedGuess = maybeYellow.containsSuch(g)(!_.isEmpty)
                        
                        TagState(
                            maybeYellow
                                .mapIf(!matched && !matchedCurrent)(_.modifyOrAdd(c)(List(index), _ :+ index))
                                .mapIf(!matched && matchedGuess)(_.modify(g)(_.tail)),
                            occurrences 
                                .mapIf(!matched && !matchedGuess)(_.modifyOrAdd(g)(1, _ + 1))
                                .mapIf(!matched && matchedCurrent)(_.modify(c)(_ - 1)),
                            tagged
                                .mapAs(_ :+ (if (matched) (c, Green) else if (matchedCurrent) (c, Yellow) else (c, Gray)))
                                .mapIf(!matched && matchedGuess)(_.updated(maybeYellow(g).head, g -> Yellow)),
                            index + 1)
                    }
                }//.inspect(println(_))
            }.inspect(_.tagged)
    }    
}

In [ ]:
run(new TagSpec(Wordle.TagState.tag))

In [ ]:
Wordle.console.unsafeRunSync

### Third version: ultra-declarative

In [ ]:
implicit class ListFoldLeftM[A](list: List[A]){
    def foldLeftM[B, M[_]: Monad](initial: B)(step: (B, A) => M[B]): M[B] = {
        def foldLeftM_Aux(initial: M[B])(step: (B, A) => M[B]): M[B] = 
            list match {
                case List() => initial
                case head :: tail => 
                    initial >>= { b => 
                        foldLeftM_Aux(step(b, head))(step)
                    }
            }
        
        foldLeftM_Aux(initial.pure[M])(step)
    }
}

In [ ]:
type TagStateState[A] = State[TagState, A]

case class TagState(maybeYellow: Map[Char, List[Int]], 
                    occurrences: Map[Char, Int],
                    index: Int)
object TagState{
/*
    def tag(word: String, guess: String): TaggedWord = 
        tagState(word, guess).runA(initial).value
*/
    val initial = TagState(Map(), Map(), 0)

    def tag(word: String, guess: String): TaggedWord = 
        word.zip(guess).toList.foldLeftM[TaggedWord, TagStateState](IndexedSeq()){
            case (tagged, (c, g)) => State{ 
                case TagState(maybeYellow, occurrences, index) => {
                    val matched = c == g
                    val matchedCurrent = occurrences.containsSuch(c)(_ > 0)
                    val matchedGuess = maybeYellow.containsSuch(g)(!_.isEmpty)
                    
                    (TagState(
                        maybeYellow
                            .mapIf(!matched && !matchedCurrent)(_.modifyOrAdd(c)(List(index), _ :+ index))
                            .mapIf(!matched && matchedGuess)(_.modify(g)(_.tail)),
                        occurrences 
                            .mapIf(!matched && !matchedGuess)(_.modifyOrAdd(g)(1, _ + 1))
                            .mapIf(!matched && matchedCurrent)(_.modify(c)(_ - 1)),
                        index + 1),
                     
                     tagged.mapAs(_ :+ (if (matched) (c, Green) else if (matchedCurrent) (c, Yellow) else (c, Gray)))
                           .mapIf(!matched && matchedGuess)(_.updated(maybeYellow(g).head, g -> Yellow)))   
                }
            }
        }.runA(initial).value
}    

In [ ]:
run(new TagSpec(TagState.tag))

### Fourth version: plain foldLeft

In [13]:
type TagStateState[A] = State[TagState, A]

case class TagState(maybeYellow: Map[Char, List[Int]], 
                    occurrences: Map[Char, Int],
                    tagged: TaggedWord,
                    index: Int)
object TagState{
/*
    def tag(word: String, guess: String): TaggedWord = 
        tagState(word, guess).runA(initial).value
*/
    val initial = TagState(Map(), Map(), IndexedSeq(), 0)

    def tag(word: String, guess: String): TaggedWord = 
        word.zip(guess).toList.foldLeft(initial){
            case (TagState(maybeYellow, occurrences, tagged, index), (c,g)) => {
                val matched = c == g
                val matchedCurrent = occurrences.containsSuch(c)(_ > 0)
                val matchedGuess = maybeYellow.containsSuch(g)(!_.isEmpty)

                TagState(
                    maybeYellow
                        .mapIf(!matched && !matchedCurrent)(_.modifyOrAdd(c)(List(index), _ :+ index))
                        .mapIf(!matched && matchedGuess)(_.modify(g)(_.tail)),
                    occurrences 
                        .mapIf(!matched && !matchedGuess)(_.modifyOrAdd(g)(1, _ + 1))
                        .mapIf(!matched && matchedCurrent)(_.modify(c)(_ - 1)),
                    tagged
                        .mapAs(_ :+ (if (matched) (c, Green) else if (matchedCurrent) (c, Yellow) else (c, Gray)))
                        .mapIf(!matched && matchedGuess)(_.updated(maybeYellow(g).head, g -> Yellow)),
                    index + 1)
            }
        }.tagged
}    

defined type TagStateState
defined class TagState
defined object TagState

In [14]:
run(new TagSpec(TagState.tag))

cmd9$Helper$TagSpec:
matching characters
- should be detected
non-matching characters
- should be detected
misplaced characters
- should be detected
